# R7 — Image Caption & Alt-Text Generator
**Generative AI Remedial Assignment**

This prototype uses the pretrained BLIP image-captioning model. Upload an image and the model generates a short caption. The app also presents a concise alt-text version for accessibility.

**Important:** This notebook downloads a pretrained model from Hugging Face, so Colab needs an internet connection. No model training is required for this prototype.

In [1]:
!pip -q install transformers gradio pillow

In [2]:
# 2. Import libraries and check runtime
import torch
from PIL import Image
from transformers import BlipProcessor, BlipForConditionalGeneration
import gradio as gr

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Runtime device:", device)

Runtime device: cuda


In [3]:
# 3. Download/load the pretrained BLIP model
MODEL_NAME = "Salesforce/blip-image-captioning-base"

processor = BlipProcessor.from_pretrained(MODEL_NAME)
model = BlipForConditionalGeneration.from_pretrained(MODEL_NAME).to(device)
model.eval()

print("BLIP model loaded successfully.")

preprocessor_config.json:   0%|          | 0.00/287 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/4.56k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/506 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

... (2 outputs left out)

Loading weights:   0%|          | 0/473 [00:00<?, ?it/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  990MB            

model.safetensors: downloading bytes:           |  0.00B            

BLIP model loaded successfully.


In [4]:
# 4. Define caption and alt-text generation
def generate_caption_and_alt_text(image):
    if image is None:
        return "Please upload an image.", ""

    if not isinstance(image, Image.Image):
        image = Image.fromarray(image)

    image = image.convert("RGB")
    inputs = processor(images=image, return_tensors="pt").to(device)

    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=40,
            num_beams=4
        )

    caption = processor.decode(output[0], skip_special_tokens=True).strip()
    alt_text = caption[0].upper() + caption[1:] if caption else "Image content could not be described."
    return caption, alt_text

print("Caption-generation function is ready.")

Caption-generation function is ready.


## Output 1 — Test with one image

Run the next cell, upload a photo, and inspect the generated caption and alt-text. Capture a screenshot for your report.

In [ ]:
# 5. Test one image
from google.colab import files
uploaded = files.upload()

for filename in uploaded:
    test_image = Image.open(filename).convert("RGB")
    display(test_image)
    caption, alt_text = generate_caption_and_alt_text(test_image)
    print("Generated caption:", caption)
    print("Alt-text:", alt_text)

## Output 2 — Launch the image-captioning application

Run this cell. Colab/Gradio will display a public demo link. Open it, upload an image, and click Submit. Capture a screenshot showing the app and its result.

In [ ]:
# 6. Build and launch the Gradio app
def caption_app(image):
    caption, alt_text = generate_caption_and_alt_text(image)
    return caption, alt_text

demo = gr.Interface(
    fn=caption_app,
    inputs=gr.Image(type="pil", label="Upload an image"),
    outputs=[
        gr.Textbox(label="Generated caption"),
        gr.Textbox(label="Short alt-text")
    ],
    title="AI Image Caption & Alt-Text Generator",
    description="Upload an image to generate a caption and concise alt-text using the pretrained BLIP model."
)

demo.launch(share=True)

## Output 3 — Test on 20 images

The assignment asks for testing on 20 images with a correctness note. Use 20 different images (for example, your own photographs or permitted sample images). Upload each image in the app and record the output in the table below. Do not invent results.

| No. | Image description | Generated caption | Correct / Partly correct / Incorrect | Notes |
|---|---|---|---|---|
| 1–20 | Fill after testing | Fill after testing | Fill after testing | Fill after testing |

For each image, compare the caption with the visible content. A caption can be grammatically correct but still miss an important object or detail.

## Limitations

- The model may miss small objects or background details.
- Captions may be too general or occasionally incorrect.
- Performance can vary with image quality, lighting, and unusual scenes.
- The alt-text in this prototype is based on the generated caption; it is not a separate independently trained model.

## Conclusion

This prototype demonstrates image captioning using a pretrained BLIP model. It accepts an image and generates a natural-language caption and concise alt-text. Testing on different images helps evaluate the usefulness and limitations of the generated descriptions.